# S44_04 — Positional Encoding

## The problem

Attention is **permutation-invariant** — it treats the input as a *set*, not a *sequence*. Swapping tokens 3 and 7 gives identical attention scores. But word order matters: "dog bites man" ≠ "man bites dog".

Positional encodings inject order information by adding a position-dependent signal to the token embeddings.

## Approach 1 — Sinusoidal (original transformer)

Vaswani et al. (2017) used fixed sinusoids at different frequencies:

In [ ]:
import torch
import math
import matplotlib.pyplot as plt

def sinusoidal_encoding(max_len, d_model):
    PE = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()  # (max_len, 1)
    div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
    PE[:, 0::2] = torch.sin(pos * div)   # even dims: sin
    PE[:, 1::2] = torch.cos(pos * div)   # odd dims: cos
    return PE

PE = sinusoidal_encoding(100, 64)
plt.figure(figsize=(10, 3))
plt.imshow(PE.T, aspect='auto', cmap='RdBu', origin='lower')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Embedding dimension')
plt.title('Sinusoidal positional encoding')
plt.tight_layout()
plt.show()

Properties of sinusoidal encoding:
- Each position has a unique pattern
- $PE[pos+k]$ is a linear function of $PE[pos]$ — relative positions are learnable
- Generalises to sequence lengths **longer than seen during training** (unlike learned embeddings)

## Approach 2 — Learned positional embeddings

Most models (BERT, GPT-2, Llama 2) simply learn an `nn.Embedding(max_seq_len, d_model)`. Simpler, slightly better in practice, but can't generalise beyond `max_seq_len`.

## Approach 3 — Rotary Position Embeddings (RoPE) ← current standard

Used by Llama, Mistral, Qwen, Gemma, and most modern open models. RoPE encodes *relative* positions by **rotating** Q and K vectors before computing attention:

In [ ]:
def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, seq_len, d_k, base=10000):
    """Minimal RoPE — rotates Q and K by position-dependent angles."""
    pos = torch.arange(seq_len, dtype=torch.float32)
    theta = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
    freqs = torch.outer(pos, theta)                         # (seq, d_k//2)
    cos = freqs.cos().repeat_interleave(2, dim=-1)          # (seq, d_k)
    sin = freqs.sin().repeat_interleave(2, dim=-1)          # (seq, d_k)

    q_rot = q * cos + rotate_half(q) * sin
    k_rot = k * cos + rotate_half(k) * sin
    return q_rot, k_rot

# Advantages of RoPE:
# - Encodes *relative* distance between tokens directly in attention scores
# - Can be extended to longer contexts than seen during training (YaRN, LongRoPE)
# - No extra parameters
print('RoPE: relative positional encoding with no learned parameters')

## Summary

| Method | Introduced in | Used by | Extrapolates? |
|--------|--------------|---------|---------------|
| Sinusoidal (absolute) | Original transformer | — | Yes, but degrades |
| Learned absolute | BERT, GPT-2 | BERT, GPT-2 | No |
| ALiBi (relative) | 2022 | MPT, BLOOM | Yes |
| **RoPE** (relative) | 2021 | **Llama, Mistral, Qwen, Gemma** | Yes (with tricks) |

Next: [S44_05_bert_gpt_t5_architectures.ipynb](./S44_05_bert_gpt_t5_architectures.ipynb)
